# k3-e5 — Task C1: Mã hoá mE5-large cho đoạn cha + đoạn con (GPU T4 ×2 độc lập của Role C)

**Cài đặt:** Accelerator = **GPU T4 ×2**, Internet = On.
**Add Input:** `mir2-code` (bản mới nhánh c-e5), `mir2-data`, `mir2-chunks` (output k1 của team).

**Tài nguyên & Phân việc Role C:**
- Role C chạy độc lập trên 1 phiên Kaggle 2 GPU T4 (`MEMBERS = 1`, `MEMBER = 0`).
- GPU 0 chạy shard `0/2`, GPU 1 chạy shard `1/2` $\rightarrow$ Chạy song song hoàn thành 100% kho của C.
- **Mốc Go/No-Go (Hạn chót 21/10):** Ô (2) chạy lệnh `profile` đo thời gian mẫu thực tế và đưa ra quyết định tự động.
- **Fallback:** Nếu NO-GO (nguy cơ quá 30h quota hoặc trễ hạn 24/10), đổi sang `LEVELS = ['parents']` (Option A).
**Đầu ra:** `emb_e5/{parents,children}/<khối>/` $\rightarrow$ Tạo dataset **`mir2-emb-e5`**.

In [ ]:
# (1) Chuẩn bị môi trường & cấu hình mE5-large
NB_VERSION = 'k3-e5-v1 (20/10/2026)'
MEMBER = 0           # Role C chạy độc lập: 1 thành viên xử lý toàn bộ kho
MEMBERS = 1          # 2 GPU chạy shard 0/2 và 1/2
LEVELS = ['parents', 'children']  # Nếu NO-GO -> đổi thành ['parents']
SKIP_FROM = []
RUN_PROFILE = True   # Bật đo tốc độ & đánh giá Go/No-Go
QUOTA_GPU_HOURS = 30.0

import glob, os, shutil, sys, zipfile
KAGGLE = os.environ.get('KAGGLE_ROOT', '/kaggle')
CODE = f'{KAGGLE}/working/code'
shutil.rmtree(CODE, ignore_errors=True)

# 1. Ưu tiên giải nén mir2_code.zip mới nhất của huyphong09 (Task C1: mE5-large + profile)
zips = sorted(glob.glob(f'{KAGGLE}/input/**/mir2_code.zip', recursive=True))
hp_zips = [z for z in zips if 'huyphong09' in z.replace('\\', '/')]
best_zip = hp_zips[-1] if hp_zips else (zips[-1] if zips else None)

extracted = False
if best_zip:
    print(f'-> Đang giải nén mã nguồn Task C1 từ file zip: {best_zip}')
    zipfile.ZipFile(best_zip).extractall(CODE)
    extracted = True
else:
    hits = sorted(glob.glob(f'{KAGGLE}/input/**/mirret/__init__.py', recursive=True), key=len)
    hp_hits = [h for h in hits if 'huyphong09' in h.replace('\\', '/')]
    best_hit = hp_hits[0] if hp_hits else (hits[0] if hits else None)
    if best_hit:
        src_dir = os.path.dirname(os.path.dirname(best_hit))
        print(f'-> Sao chép mã nguồn từ thư mục unzipped: {src_dir}')
        shutil.copytree(src_dir, CODE)
        extracted = True

if not extracted:
    raise AssertionError("Chưa thêm dataset mir2-code (Vào Add Input -> Your Datasets -> huyphong09/mir2-code)")

# Kiểm tra xem code nạp vào đã có tính năng profile cho Task C1 chưa
with open(f'{CODE}/mirret/encode.py', 'r', encoding='utf-8') as f:
    code_text = f.read()
if 'profile' not in code_text:
    raise RuntimeError(
        "PHÁT HIỆN CODE BẢN CŨ! Bạn đang nạp nhầm 'mir2-code' bản cũ (07/10).\n"
        "Hãy kiểm tra cột Input bên phải:\n"
        "  1. Xoá dataset 'nhntrnduy/mir2-code' (hoặc 'mir-code' cũ).\n"
        "  2. Bấm '+ Add Input' -> Chọn tab 'Your Work' / 'Your Datasets' -> Thêm 'huyphong09/mir2-code'."
    )

os.chdir(CODE); sys.path.insert(0, CODE)
from mirret.kaggle_nb import setup, sh, par, locate, locate_all, report
K = setup(NB_VERSION, {
    'encode.model': 'intfloat/multilingual-e5-large',
    'encode.out_dir': 'emb_e5'
}, inputs=('chunks',))

In [ ]:
# (2) Speed Profiling & Đánh giá mốc Go/No-Go
prev = [str(d / 'emb_e5') for d in locate_all('emb_e5')]
print('emb_e5 trong input:', prev)
old = [p for p in prev if any(f'/{x}/' in p + '/' for x in SKIP_FROM)]
mine_prev = [p for p in prev if p not in old]
skip = f' --skip {",".join(old)}' if old else ''
resume = f' --resume {",".join(mine_prev)}' if mine_prev else ''
print('bỏ qua (bản cũ):', old, '| chạy tiếp từ:', mine_prev)

if RUN_PROFILE:
    print('=== [TASK C1] CHẠY SPEED PROFILING & ĐÁNH GIÁ GO / NO-GO ===')
    sh(f'python -m mirret.encode profile --config {K.cfg} --device cuda:0 --n-children 2000 --n-parents 1000 --quota {QUOTA_GPU_HOURS}')

for lv in LEVELS:
    sh(f'python -m mirret.encode plan --config {K.cfg} --level {lv}{skip}')

In [ ]:
# (3) Chạy mã hoá 100% Corpus trên 2 GPU T4 song song
# GPU 0: Shard 0/2; GPU 1: Shard 1/2
n = 2 * MEMBERS
for lv in LEVELS:
    par([f'CUDA_VISIBLE_DEVICES={g} python -m mirret.encode run --config {K.cfg} --level {lv} '
         f'--shard {g}/{n} --device cuda:0{skip}{resume}' for g in range(2)], f'encode_{lv}')
report('k3_e5')